<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# CP101: Assignment 3 - HealthFirst NYC: Guiding Safe Dining Choices in NYC

**Due Date:** Wed, Oct 21, 2026 - 11:59 pm

**Total Points:** 100 (+ 20 bonus)

---

### Instructions
- Complete all questions in this notebook and follow all instructions carefully.
- Make sure you use the exact variable and file names indicated here.
- For each analysis step, document your observations by presenting relevant counts, summary statistics, and key descriptive metrics.
- Include comments to explain your solutions.
- The `...` in the code cells serve as placeholders for your code. We have included helper code and comments to guide your analysis. Replace these placeholders with your solutions.
- Run **Kernel → Restart & Run All** before submitting to ensure your code works.
- Save your work as you go and before you submit.
- Submit the autograder zip (coding questions) and the written-work PDF (manual questions) to their respective Gradescope portals.

---

### Context

You are a Data Analyst at HealthFirst NYC, working closely with the NYC Department of Health. Your mission is to help ensure food safety across the city by carefully analyzing restaurant inspection data. This analysis will support city inspectors, inform public policy, and guide residents' dining decisions.

The NYC restaurant inspection dataset ([access here](https://data.cityofnewyork.us/Health/DOHMH-New-York-City-Restaurant-Inspection-Results/43nn-pn8j/about_data)) provides detailed information about health grades, violations, and locations across the city. Your objective is to clean, analyze, and extract actionable insights from this dataset. Before beginning, review the dataset and the accompanying data dictionary, [access here](https://data.cityofnewyork.us/api/views/43nn-pn8j/files/ec33d2c8-81f5-499a-a238-0213a38239cd?download=true&filename=RestaurantInspectionDataDictionary_09242018.xlsx), to understand the meaning of each column, possible values, and any inherent limitations.

> **Note:** As the dataset is compiled from several administrative systems, some illogical values (due to data entry or transfer errors) may appear, and data might be missing. For more details on grading, please consult the [NYC Restaurant Grading FAQ](https://a816-health.nyc.gov/ABCEatsRestaurants/#!/faq).

Your goal is to incrementally build actionable insights from raw data to meaningful conclusions. This notebook walks you through data preparation, cleaning, spatial join, grade validation, and temporal analysis. Follow the instructions carefully and ensure your variable names match those specified.

### Provided Files

- **`data/raw/NYC_Restaurant_Inspection_1824.csv`**: NYC restaurant inspection records.
- **`data/raw/nyc_neighborhoods.parquet`**: NYC neighborhood (sub-borough) boundaries as polygons.

> **About the inspection file.** The full published dataset is close to 100 MB, which is too large to hand out and slow to work with. To make the file size manageable we did three things: we **removed a column** (`VIOLATION DESCRIPTION`, a long block of free text this assignment never uses), we **decreased the number of rows** (a sample of inspections, sampled whole so you never see part of one inspection), and we dropped Staten Island. The one thing we did *not* remove is the messy data: every record with a missing coordinate, an impossible location, or a grade that contradicts its own score is still in the file. The problems you find in Part B are real errors in the City's data, not errors we planted for you.

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("CP101_A3.ipynb")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Part A: Data [5 points]

Before we can analyze trends or compute metrics, we must load the raw restaurant inspection data and inspect data quality and sanity.

### Task A.1: Load the Restaurant Inspection Data

- Import the necessary libraries.
- Load the `data/raw/NYC_Restaurant_Inspection_1824.csv` file into a pandas DataFrame named **`restaurants_df`**.
- Display the first few rows.

Notice what kind of file this is. It is a plain `.csv` table: it has a `Latitude` column and a `Longitude` column, but nothing in the file marks it as spatial data. Pandas is the right tool to read it, and it is the right tool for exactly one step. In Task A.2 you will turn those two coordinate columns into real geometry.

In [ ]:
# Import necessary libraries
import pandas as pd
import geopandas as gpd

# Load the restaurant inspection data
file_path = "data/raw/NYC_Restaurant_Inspection_1824.csv"
restaurants_df = ...

In [ ]:
grader.check("a1")

Preview the data before continuing:

In [ ]:
restaurants_df.head()

### Task A.2: Create a GeoDataFrame

`restaurants_df` is still an ordinary table. To do anything spatial with it (mapping it, or joining it to neighborhood polygons in Part C) it needs a **geometry** column: one point per row, built from the coordinate columns.

- Use the `Longitude` and `Latitude` columns from **`restaurants_df`** to create a GeoDataFrame named **`restaurants_gdf`**, with CRS set to **EPSG:4326**.
- Display the first few rows and confirm a `geometry` column now exists.

Two details that trip people up:

- `gpd.points_from_xy()` takes **x first, then y**, which means **longitude first, then latitude**. This is the opposite of the order people usually say out loud ("lat/long"), and swapping them lands every restaurant in the wrong hemisphere.
- Setting `crs="EPSG:4326"` does not convert anything. It records what the numbers already mean: degrees of longitude and latitude. You are labeling the data, not transforming it.

In [ ]:
restaurants_gdf = ...

In [ ]:
grader.check("a2")

> #### Aside: could GeoPandas have read the `.csv` directly? (not graded)
>
> Reasonable question, and the answer is yes, with a catch worth understanding.
>
> GeoPandas can hand the file to GDAL and ask it to build the points during reading:
>
> ```python
> restaurants_gdf = gpd.read_file(
>     file_path, engine="pyogrio",
>     X_POSSIBLE_NAMES="Longitude", Y_POSSIBLE_NAMES="Latitude",
>     KEEP_GEOM_COLUMNS="NO", AUTODETECT_TYPE="YES",
> ).set_crs("EPSG:4326")
> ```
>
> We do not teach this as the default, for three reasons. Without those options the *same* call quietly returns a plain `DataFrame` with no geometry at all and no warning. The result has no CRS, so you still have to call `.set_crs()` yourself. And without `AUTODETECT_TYPE="YES"` every column arrives as text, so `score` would be the string `"21"` rather than the number `21`.
>
> There is also a difference that matters for the cleaning you are about to do in Part B. Some rows in this file have no coordinates:
>
> | | a row with a missing `Latitude` becomes | does `.isna()` flag it? |
> |---|---|---|
> | `gpd.points_from_xy` | `POINT (-73.9 NaN)` | **No** |
> | `gpd.read_file` with the options above | `None` | Yes |
>
> `points_from_xy` gives you a point object that *exists* but whose coordinate is `NaN`. It is not missing and it is not empty, so no missing-value check will catch it, but it is not a valid location either: it will silently corrupt `.total_bounds`, it plots nowhere, and after `.to_crs()` it degrades to `POINT (NaN NaN)`.
>
> That is precisely why the first cleaning step in Part B is to drop rows with no `latitude` or `longitude`, **before** any spatial work happens. Run the cell below to see how many such rows are hiding in this dataset.

In [ ]:
# Not graded -- run this to see the two kinds of bad location in the raw data.
n_nan_coords = restaurants_gdf["geometry"].get_coordinates().isna().any(axis=1).sum()
print(f"Rows whose point holds a NaN coordinate: {n_nan_coords}")
print(f"...of which .isna() flags as missing:    {restaurants_gdf['geometry'].isna().sum()}")

# A second, sneakier problem: coordinates recorded as exactly (0, 0).
null_island = (restaurants_gdf["Longitude"] == 0) & (restaurants_gdf["Latitude"] == 0)
print(f"\nRows sitting at exactly (0, 0): {null_island.sum()}")
print("(0, 0) is a real, valid point in the Atlantic Ocean off West Africa, often")
print("called Null Island. No missing-value check will ever flag it -- only knowing")
print("that these restaurants are supposed to be in New York City will.")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Part B: Data Cleaning, Standardization, and Sanity Checks [24 points]

Accurate analysis depends on clean, consistent data. In this step, we standardize column names and ensure numeric and date columns have the correct types. This answers: "Is our data in a reliable format for further analysis?"

### Task B.1: Standardize Column Names and Convert Data Types

#### 1. Standardize column names

- Create a GeoDataFrame named `cleaned_gdf` from `restaurants_gdf`. Use `cleaned_gdf` for the rest of the assignment.
- Rename all columns in `cleaned_gdf` so they use **lowercase letters** and **underscores** (e.g. `INSPECTION DATE` becomes `inspection_date`).

#### 2. Drop restaurants with missing critical data

- **Count missing values per restaurant:** create a Series named `missing` that references the number of missing values in each row across all columns. The Series should use the **row index** from `cleaned_gdf`, and each value should be the number of missing entries for that restaurant.
- **Remove restaurants without location data:** find rows where `latitude` or `longitude` is missing and remove them from `cleaned_gdf`. Either coordinate missing on its own is enough to make the location unusable, so a row needs *both* to survive.
- **Remove restaurants missing key inspection details:** find rows where `score`, `inspection_type`, and `cuisine_description` are all missing at the same time, and remove those rows from `cleaned_gdf`.

#### 3. Check and convert data types

- **Inspect current data types:** write code to display the data type of every column in `cleaned_gdf`.
- **Convert numeric columns where needed:** look at the types of `score`, `latitude`, and `longitude`. If any did not arrive as numeric types, convert them using a vectorized method such as `pd.to_numeric(..., errors="coerce")`.
- **Convert date columns where needed:** look at the types of `inspection_date`, `grade_date`, and `record_date`. Convert them to datetime using `pd.to_datetime(..., errors="coerce")`.
- **Then look at what you converted.** `errors="coerce"` turns unparseable text into `NaT`, which is easy to spot. It does *not* protect you from text that parses perfectly well and still is not a real date: this file uses `01/01/1900` as a placeholder for "no inspection date recorded", and `pd.to_datetime` will hand you back a clean, valid timestamp for every one of them. Count how many rows land in 1900 (`(cleaned_gdf["inspection_date_dt"].dt.year == 1900).sum()`). A successful conversion is not the same as a correct value.
- **Add the converted values as new columns:** assign each converted result to a new column whose name ends with `_dt`. For example, `inspection_date` becomes `inspection_date_dt`.
- **Check for other needed conversions:** review the remaining columns and convert any additional column whose current type does not match the kind of data it holds.

> **Note:** because this file is a `.csv`, it carries no type information of its own; every column arrives as whatever pandas guessed from the text. This is exactly why step 3 exists. To make sure your missing data types are handled correctly, refer to the [W6S2 slides](https://drive.google.com/file/d/1L97hcnzdh1AuqisTipbSt_6nHCYto0Hj/view?usp=sharing).

In [ ]:
# Create a copy for cleaning
cleaned_gdf = ...

# 1. Standardize column names


# 2. Drop restaurants with missing critical data
missing = ...


# 3. Check current data types, then convert numeric and date columns
#    (store each converted date in a new column ending in _dt)


# Display the cleaned GeoDataFrame
cleaned_gdf.head()

In [ ]:
grader.check("b1")

### Task B.2: Sanity Check and Correct Inspection Grades

**Background:** NYC inspection grades are defined by these guidelines:

- **"A" grade:** 0 to 13 sanitary violation points
- **"B" grade:** 14 to 27 sanitary violation points
- **"C" grade:** 28 or more sanitary violation points

**Report inconsistencies:**

1. Create masks to filter the GeoDataFrame based on scores, one mask for each grade (`mask_a`, `mask_b`, `mask_c`).
2. Write the code to find rows where the `grade` column does not match the grade the score implies. For example, a row with a score of 30 that is labeled `B` is inconsistent, because 30 points implies a `C`.
3. Point a variable named `inconsistencies` at the rows whose A, B, or C grade does not match their expected grade. `inconsistencies` should be a DataFrame with the same index and three columns:
   - `score`
   - `grade`
   - `expected_grade`
4. Point `n_grading_inconsistencies` at its length, and `print` that.

**Two rules that decide your count, so read them before you write the mask:**

- **Grades of `Z`, `P`, or `N` are never inconsistencies.** Those codes mean the grade is
  pending or was never issued, not that it disagrees with the score.
- **A row with an A, B, or C grade but no score *is* an inconsistency.** A posted grade that
  cannot be checked against anything is exactly the kind of record a health department needs
  flagged. There are only three such rows, and they are easy to lose: `pd.NA` never compares
  equal to anything, so `grade != expected_grade` already returns `True` for them. That is
  the behaviour you want here, but notice it is a side effect of how missing values compare,
  not something you asked for. Confirm it rather than assume it.

In [ ]:
# One mask per grade band, based on the score
mask_a = ...
mask_b = ...
mask_c = ...

# The grade each score implies
inconsistencies = ...
n_grading_inconsistencies = ...
print(n_grading_inconsistencies)

### BONUS: Correct the Grades [10 extra points]

Create a new column named `fixed_grade` in `cleaned_gdf` that corrects the inconsistencies you just found. For rows where the posted grade disagrees with the score, `fixed_grade` should be the grade the score implies. For example, a row with a score of 30 posted as `B` should get a `fixed_grade` of `C`.

Name the column exactly `fixed_grade`. How you get there is up to you.

**You will not be able to fix all 643.** Three of the rows you flagged in B.2 carry an A, B,
or C grade with **no score at all**. You flagged them because a grade nobody can check is a
real problem, but there is nothing to correct them *to*: inventing a letter to replace a
missing score would be worse than leaving the record as it stands. Correct only the rows whose
score actually implies a grade, and leave those three exactly as posted. Your `fixed_grade`
should therefore differ from `grade` in **640** rows, not 643.

"I found 643 problems and can responsibly fix 640 of them" is a better result than silently
fixing all 643.

**Important:** do not modify rows where the grade is `Z`, `P`, or `N`. Those indicate pending or not-yet-graded statuses, and overwriting them would invent a grade the inspector never issued. Rows that were already correct should keep the grade they have.

In [ ]:
# bonus code here

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Part C: Neighborhood Level Analysis [33 points]

In New York City, neighborhood-level trends play a crucial role in understanding food safety and targeting health inspections. By linking each restaurant to its corresponding neighborhood, city officials can identify areas with consistently high violation scores, spot emerging health risks, and allocate resources more effectively.

### Task C1: Spatial Join Restaurants with Their Neighborhoods

This step ensures that every restaurant is assigned to the correct neighborhood so we can later analyze patterns at a finer geographic scale. To put it simply, we want each restaurant in our data (from `cleaned_gdf`) to "pick up" the neighborhood information if the restaurant's location falls within that neighborhood's boundaries.

Over the next two questions you will load the neighborhood boundaries, measure each
neighborhood's area, and join the restaurants to them. **All of the code lives in the cell
under the next question.** This first question is a short reasoning question to answer before
you write it.

Two things you will need to know to write that code:

- The boundaries ship in **EPSG:4326**, whose units are degrees.
- Area has to be measured in a **projected** CRS. For New York City, `EPSG:2263` is the
  conventional choice.

**Question:** once you have reprojected to `EPSG:2263` and computed `neigh_area` in the next
question, what will its units be, and why would computing `.area` *before* reprojecting give
you a meaningless number?

This is graded manually, so write your answer in the cell below rather than as a `print`.

Now write the code for the whole of Task C1:

1. Load the `data/raw/nyc_neighborhoods.parquet` file into a GeoDataFrame named `neighs_gdf`. Unlike the inspections `.csv`, this file stores its geometry and its CRS internally, so `gpd.read_parquet()` gives you a GeoDataFrame directly with no coordinate columns to assemble.
2. Reproject `neighs_gdf` to `EPSG:2263`, then create a new column named `neigh_area` giving the area of each neighborhood. (This is the column you reasoned about in the previous question, so check that its units match what you predicted.)
3. Create a GeoDataFrame named `restaurants_neighs_gdf` by performing a spatial join between **`cleaned_gdf`** and **`neighs_gdf`**. (**Hint:** use `gpd.sjoin` with `how="left"` and `predicate="within"`.)
4. Point a variable named `neighs_restaurant_counts` at the number of restaurants per neighborhood, indexed by neighborhood name.

**Hint:** a spatial join requires both GeoDataFrames to be in the *same* CRS. Your restaurants are in EPSG:4326 and your neighborhoods are now in EPSG:2263, so reproject one to match the other before joining.

Using `how="left"` means every restaurant is kept even if it does not fall inside any neighborhood polygon. That is deliberate: the count of unmatched restaurants is itself a data-quality signal, and you saw in Part A why some points will never land anywhere sensible.

In [ ]:
# Load the neighborhoods data
neigh_path = "data/raw/nyc_neighborhoods.parquet"
neighs_gdf = ...

# Check the CRS, reproject if needed, then compute the area


# Perform a spatial join between cleaned_gdf and neighs_gdf
restaurants_neighs_gdf = ...

# Count the restaurants per neighborhood
neighs_restaurant_counts = ...

In [ ]:
grader.check("c1_join")

### Task C2: Aggregate and Map Neighborhood Inspection Scores

In this part, we compute the average, maximum, and minimum inspection scores for each neighborhood to detect high-risk areas and understand local variations in restaurant health violations.

To make these insights more actionable, we will visualize them using choropleth maps, which provide an intuitive way to see which neighborhoods consistently have higher violation scores. This supports data-driven decision-making in food safety monitoring across the city.

1. Create a DataFrame named `neighs_stats_df` that, for each neighborhood, gives:
   - Average inspection score (`avg_score`)
   - Maximum inspection score (`max_score`)
   - Minimum inspection score (`min_score`)

   **Hint:** use `groupby("neigh_name")` and `.agg()`. Note that this operation produces a pandas DataFrame, not a GeoDataFrame: the geometry does not survive a `groupby`.

2. To prepare the data for mapping:
   - Add the `geometry` column back by merging `neighs_stats_df` with `neighs_gdf` on the `neigh_name` column.
   - Create a GeoDataFrame named `neighs_stats_gdf` from the merged result.
   - Use `.set_geometry("geometry")` to set the active geometry column.

The plotting code is written for you in the next question. You only need to fill in the placeholders.

In [ ]:
neighs_stats_df = ...
neighs_stats_gdf = ...

Create two static choropleth maps to visualize:

- The average inspection score per neighborhood, saved as `avg_score_map.png`.
- The maximum inspection score per neighborhood, saved as `max_score_map.png`.

The visualization code below is provided. **Only replace the `...` placeholders.** The `raise` statements are there on purpose: they turn the most common mistakes in this step (passing a plain DataFrame because you forgot `.set_geometry()`, or misspelling a column name) into a clear error message instead of a confusing one.

In [ ]:
# VISUALIZATION CODE -- ONLY REPLACE WHERE THERE IS ...
import matplotlib.pyplot as plt

## Plot #1 - Average inspection score per neighborhood
gdf_to_plot = ...
col_to_plot = ...

if not isinstance(gdf_to_plot, gpd.GeoDataFrame):
    raise TypeError("Input must be a GeoDataFrame, did you forget setting the geometry column?")
if col_to_plot not in gdf_to_plot.columns:
    raise ValueError(f"Column '{col_to_plot}' not found in the GeoDataFrame.")
if gdf_to_plot.empty:
    raise ValueError("GeoDataFrame is empty. Cannot generate a choropleth map.")

fig, ax = plt.subplots(figsize=(8, 8))
gdf_to_plot.plot(column=col_to_plot, legend=True, ax=ax)
ax.set_title(f"{col_to_plot} by neighborhood")
plt.axis("off")
fig.savefig(f"{col_to_plot}_map.png", dpi=200, bbox_inches="tight")
plt.show()

## Plot #2 - Maximum inspection score per neighborhood
gdf_to_plot = ...
col_to_plot = ...

if not isinstance(gdf_to_plot, gpd.GeoDataFrame):
    raise TypeError("Input must be a GeoDataFrame, did you forget setting the geometry column?")
if col_to_plot not in gdf_to_plot.columns:
    raise ValueError(f"Column '{col_to_plot}' not found in the GeoDataFrame.")
if gdf_to_plot.empty:
    raise ValueError("GeoDataFrame is empty. Cannot generate a choropleth map.")

fig, ax = plt.subplots(figsize=(8, 8))
gdf_to_plot.plot(column=col_to_plot, legend=True, ax=ax)
ax.set_title(f"{col_to_plot} by neighborhood")
plt.axis("off")
fig.savefig(f"{col_to_plot}_map.png", dpi=200, bbox_inches="tight")
plt.show()

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Part D: Temporal Analysis, Trends by Neighborhood [14 points]

Restaurant inspection scores can fluctuate over time due to seasonal factors (higher customer volumes in summer or holiday periods), changes in regulatory focus, or shifts in restaurant operations. By examining the average monthly inspection scores for each neighborhood, we can determine whether there are consistent patterns or anomalies that indicate when restaurants are more likely to show higher violation scores. This analysis helps answer:

**"How have inspection scores in the city's busiest neighborhoods moved over the period this dataset covers, and can we identify periods of heightened risk for targeted interventions?"**

We will focus on the five neighborhoods with the highest number of restaurants and analyze their monthly average inspection scores.

> **A note on the time range.** Do not filter by year. Plot everything, then be careful about
> how you read it, because this dataset is badly uneven. Your five neighborhoods span
> **January 2018 to March 2025**, but inspections collapse during the pandemic and only
> recover in 2021: 2018, 2019 and 2020 contribute fewer than 200 records *combined*, while
> 2022, 2023 and 2024 contribute roughly 87% of the total. 2025 is a partial year and stops
> in March, so its final points are not comparable to a full month elsewhere.
>
> Before you read anything into either end of your trend line, count how many inspections
> actually sit behind each month. A line drawn through three data points looks exactly as
> confident as one drawn through three hundred, and it is not.

### Task D:

- Point a DataFrame named `top5_neighs` at only the rows for the top 5 neighborhoods by restaurant count.
- In `top5_neighs`, create a new column named `year_month` from the `inspection_date_dt` column, representing each inspection's month and year (use `.dt.to_period("M")`).
- Group `top5_neighs` by `neigh_name` and `year_month` to compute the average inspection score for each neighborhood, and point a DataFrame named `top5_neighs_monthly_avg` at the result, with the average in a column named `avg_score`.
- Generate the line plot in the next question to show how the average inspection scores fluctuate over time for these five neighborhoods.

In [ ]:
top5_neighs = ...
top5_neighs_monthly_avg = ...

In [ ]:
grader.check("d_data")

Generate a line plot showing average monthly inspection scores over time for the top 5 neighborhoods, and save it as `monthly_scores_trend.png`.

The visualization code below is provided. **Only replace the `...` placeholders.** Note the `.dt.to_timestamp()` line: seaborn cannot plot a pandas `Period` directly, so the `year_month` column has to be converted back to a timestamp before plotting.

In [ ]:
# VISUALIZATION CODE -- ONLY REPLACE WHERE THERE IS ...
import matplotlib.pyplot as plt
import seaborn as sns

df_to_plot = ...
title_of_plot = ...

sns.set_style("whitegrid")
plt.figure(figsize=(12, 6))
df_to_plot = df_to_plot.copy()
df_to_plot["year_month"] = df_to_plot["year_month"].dt.to_timestamp()

sns.lineplot(data=df_to_plot, x="year_month", y="avg_score", hue="neigh_name",
             marker="o", linewidth=2.0)

plt.title(title_of_plot, fontsize=14)
plt.xlabel("Year-Month", fontsize=12)
plt.ylabel("Average Inspection Score", fontsize=12)
plt.xticks(rotation=45)
plt.legend(title="Neighborhood", fontsize=10)

plt.tight_layout()
plt.savefig("monthly_scores_trend.png", dpi=300)
plt.show()

### BONUS: Name the chart [4 extra points]

Look at the line plot you just produced. Five neighborhoods, one line each, one point per month, in a score range where all five overlap constantly.

You saw this exact failure in the Week 9 lab, in `line_chart_activities.ipynb`. Charts like
this have a name. Which one is it?

| | Chart type |
|---|---|
| **A** | Small multiples |
| **B** | Spaghetti chart |
| **C** | Slope chart |
| **D** | Sparkline |
| **E** | Stacked area chart |

Set `d_chart_type` to the letter of your answer, as a string (for example `"C"`).

In [ ]:
d_chart_type = ...

### BONUS: Fix it [6 extra points]

Naming the problem is the easy part. Now fix it, using two of the five rules from the Week 9
lab (`line_chart_activities.ipynb`) together:

- **Rule 5, small multiples.** Give each neighborhood its own panel instead of stacking all five in one frame. Every panel shares the same y-axis, so comparing across panels stays honest.
- **Rule 4, highlight one line and mute the rest.** Inside each panel, draw *all five* neighborhoods, but only that panel's own neighborhood in full color. The other four stay light gray in the background. This is what keeps small multiples comparable: each panel still shows the whole city, it just tells you where to look.

Also apply **rule 3**: label the highlighted line directly inside its panel instead of adding a legend. With one panel per neighborhood, a legend would just repeat the panel titles five times.

Fill in the `...` placeholders in the code below and save the figure as `monthly_scores_small_multiples.png`. The gray context loop is written for you; you are filling in the grid shape, the focus line's styling, and the direct label.

In [ ]:
# VISUALIZATION CODE -- ONLY REPLACE WHERE THERE IS ...
import matplotlib.pyplot as plt

# Seaborn cannot plot a Period, so convert back to a timestamp as before
plot_df = top5_neighs_monthly_avg.copy()
plot_df["year_month"] = plot_df["year_month"].dt.to_timestamp()
neighborhoods = sorted(plot_df["neigh_name"].unique())

# One row per neighborhood, sharing both axes so the panels stay comparable
fig, axes = plt.subplots(
    nrows=..., ncols=1, figsize=(11, 12), sharex=True, sharey=...
)
axes_flat = axes.flatten()

for i, neigh in enumerate(neighborhoods):
    ax = axes_flat[i]

    # Context: all five neighborhoods in light gray, so every panel still shows
    # the whole comparison. Low zorder keeps them behind the focus line.
    for other in neighborhoods:
        other_data = plot_df[plot_df["neigh_name"] == other]
        ax.plot(other_data["year_month"], other_data["avg_score"],
                color="#cccccc", linewidth=1, alpha=0.6, zorder=1)

    # Focus: this panel's own neighborhood, drawn on top in full color
    focus = plot_df[plot_df["neigh_name"] == neigh]
    ax.plot(focus["year_month"], focus["avg_score"],
            color=..., linewidth=..., zorder=...)

    # Direct label instead of a legend, placed inside the panel.
    # transform=ax.transAxes means (0, 0) is the bottom-left of the panel and
    # (1, 1) the top-right, so the label sits in the same spot in every panel.
    ax.text(..., ..., neigh, transform=ax.transAxes,
            color=..., fontweight="bold", fontsize=10)

    ax.set_ylabel("Avg score", fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(
    "Average monthly inspection score, five busiest NYC neighborhoods",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()
plt.savefig("monthly_scores_small_multiples.png", dpi=200, bbox_inches="tight")
plt.show()

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Grading rubric reminder

| Criteria | Points |
|---|:---:|
| Parts A-D: all files created and exercises completed | 76 |
| This question: efficient, well-structured notebook, comments and relevant typing, no clear GPT code | 24 |
| **Total** | **100** |
| Bonus: corrected grades (Part B2) | +10 |
| Bonus: named and fixed the chart (Part D) | +10 |
| **Max total** | **120** |

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit.

Make sure you have saved your notebook before running this cell. Then run this cell and click on the download link to get your submission zip.

In [ ]:
grader.export(pdf=False, force_save=True, run_tests=True)